# Aula 18 — Divisão entre treino e teste e escalonamento

**Módulo 6 — Introdução ao Machine Learning**

## Objetivos da aula

- Dividir os dados em conjuntos de treino e teste com `train_test_split`.
- Entender por que essa divisão é necessária.
- Padronizar variáveis numéricas com `StandardScaler`.

---

## 1. Recriando a base e os conjuntos `X` e `y` (Aula 17)

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 500
tipos_equipamento = ["Motor", "Bomba", "Compressor", "Ventilador"]

dados = pd.DataFrame({
    "equipamento_id": [f"EQ-{i:04d}" for i in range(1, n + 1)],
    "tipo_equipamento": np.random.choice(tipos_equipamento, size=n),
    "temperatura": np.round(np.random.normal(70, 12, size=n), 1),
    "pressao": np.round(np.random.normal(5.5, 1.3, size=n), 2),
    "vibracao": np.round(np.random.normal(2.4, 1.1, size=n), 2),
    "horas_operacao": np.random.randint(0, 10000, size=n),
})


def definir_status(linha):
    critico = (linha["temperatura"] >= 90) or (linha["vibracao"] >= 4.5) or (linha["pressao"] >= 8) or (linha["pressao"] <= 2)
    alerta = (linha["temperatura"] >= 80) or (linha["vibracao"] >= 3.5) or (linha["pressao"] >= 7) or (linha["pressao"] <= 3)
    if critico:
        return "critico"
    elif alerta:
        return "alerta"
    else:
        return "normal"


dados["status"] = dados.apply(definir_status, axis=1)
dados.to_csv("sensores_industriais.csv", index=False)

df = pd.read_csv("sensores_industriais.csv")
y = df["status"]

atributos_numericos = ["temperatura", "pressao", "vibracao", "horas_operacao"]
X_numerico = df[atributos_numericos]
tipo_codificado = pd.get_dummies(df["tipo_equipamento"], prefix="tipo", dtype=int)
X = pd.concat([X_numerico, tipo_codificado], axis=1)

print("X:", X.shape, "| y:", y.shape)


X: (500, 8) | y: (500,)


## 2. Por que dividir os dados em treino e teste

Se treinássemos o modelo com **todos** os dados e depois o avaliássemos com esses mesmos dados, estaríamos testando se ele "decorou" os exemplos — não se ele **aprendeu um padrão generalizável**. Por isso, separamos uma parte dos dados (o **conjunto de teste**) que o modelo nunca vê durante o treinamento, reservada exclusivamente para avaliação honesta ao final.

- **Treino**: usado para o modelo aprender os padrões.
- **Teste**: usado, só no final, para medir o quão bem o modelo generaliza para dados novos.

O Scikit-learn (biblioteca de Machine Learning que usaremos a partir de agora) oferece a função `train_test_split` para fazer essa divisão de forma aleatória.

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% dos dados reservados para teste
    random_state=42,     # garante que a divisão seja sempre a mesma ao reexecutar
    stratify=y,          # mantém a mesma proporção de cada status em treino e teste
)

print("Treino:", X_train.shape, "| Teste:", X_test.shape)


Treino: (400, 8) | Teste: (100, 8)


**Sobre os parâmetros:**

- `test_size=0.2`: reserva 20% dos dados para teste e 80% para treino — uma proporção comum é entre 70/30 e 80/20.
- `random_state=42`: fixa a "semente" aleatória, tornando a divisão **reprodutível** (o mesmo resultado toda vez que o código roda), assim como fizemos com `np.random.seed()` na Aula 11.
- `stratify=y`: como vimos na Aula 17, as classes de `status` são desbalanceadas. Sem `stratify`, a divisão aleatória poderia, por azar, deixar o conjunto de teste com poucos (ou nenhum) exemplo de `critico`. `stratify=y` garante que treino e teste mantenham a mesma proporção de cada classe da base original.

Vamos confirmar que as proporções se mantiveram.

In [3]:
print("Proporção de status no conjunto original:")
print(y.value_counts(normalize=True).round(3))

print("\nProporção de status no treino:")
print(y_train.value_counts(normalize=True).round(3))

print("\nProporção de status no teste:")
print(y_test.value_counts(normalize=True).round(3))


Proporção de status no conjunto original:
status
normal     0.560
alerta     0.322
critico    0.118
Name: proportion, dtype: float64

Proporção de status no treino:
status
normal     0.560
alerta     0.322
critico    0.118
Name: proportion, dtype: float64

Proporção de status no teste:
status
normal     0.56
alerta     0.32
critico    0.12
Name: proportion, dtype: float64


## 3. Por que escalonar variáveis numéricas

Observe as escalas bem diferentes dos nossos atributos numéricos: `temperatura` varia na casa das dezenas, `pressao` e `vibracao` na casa das unidades, e `horas_operacao` na casa dos milhares. Muitos algoritmos de Machine Learning (incluindo os que usam distância entre pontos, como o KNN que veremos na próxima aula) tratariam `horas_operacao` como "mais importante" apenas por ter números maiores — não porque realmente seja. O **escalonamento** (*scaling*) coloca essas medidas numéricas em uma escala comparável, para que o modelo as compare de forma mais justa.


In [4]:
print(X_train[atributos_numericos].describe().loc[["mean", "std", "min", "max"]])


      temperatura   pressao  vibracao  horas_operacao
mean    70.095000  5.672850  2.522750     4660.115000
std     12.500682  1.214922  1.096109     2835.232606
min     31.100000  1.800000 -0.810000        6.000000
max    116.200000  8.920000  5.910000     9984.000000


## 4. `StandardScaler`: padronização

O `StandardScaler` transforma cada atributo numérico para que tenha **média 0 e desvio padrão 1**, pela fórmula: `valor_padronizado = (valor - média) / desvio_padrão`. Essa é a mesma ideia do "desvio em relação à média" que já usamos na Aula 12, agora também dividida pelo desvio padrão.

As colunas 0/1 criadas pelo one-hot encoding já estão em uma escala pequena. Por isso, nesta aula vamos manter essas colunas como 0/1 e padronizar apenas as medidas numéricas.


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# fit_transform: calcula média/desvio das colunas numéricas do TREINO e aplica a transformação nele
X_train_scaled[atributos_numericos] = scaler.fit_transform(X_train[atributos_numericos])

# transform (sem fit!): aplica ao teste a MESMA transformação aprendida no treino
X_test_scaled[atributos_numericos] = scaler.transform(X_test[atributos_numericos])

print("Média das colunas numéricas escalonadas (treino):")
print(X_train_scaled[atributos_numericos].mean().round(2))

print("\nDesvio padrão das colunas numéricas escalonadas (treino):")
print(X_train_scaled[atributos_numericos].std(ddof=0).round(2))


Média das colunas numéricas escalonadas (treino):
temperatura       0.0
pressao          -0.0
vibracao         -0.0
horas_operacao    0.0
dtype: float64

Desvio padrão das colunas numéricas escalonadas (treino):
temperatura       1.0
pressao           1.0
vibracao          1.0
horas_operacao    1.0
dtype: float64


## 5. Por que `fit` só no treino? O cuidado com *data leakage*

Este é um dos erros mais comuns (e mais sérios) em Machine Learning: se calculássemos a média e o desvio padrão usando **todos** os dados (treino + teste) antes de dividir, informação do conjunto de teste "vazaria" para o processo de treinamento — isso é chamado de ***data leakage*** (vazamento de dados). O modelo pareceria funcionar melhor do que realmente funcionaria em produção, com dados genuinamente novos.

A regra é simples e vale para qualquer etapa de pré-processamento: **`fit` (ou `fit_transform`) somente no treino; no teste, sempre `transform`.**

## 6. Resumo da aula

- `train_test_split` separa os dados em treino (para aprender) e teste (para avaliar de forma honesta).
- `stratify=y` preserva a proporção das classes em treino e teste — essencial quando as classes são desbalanceadas.
- `StandardScaler` padroniza variáveis numéricas para média 0 e desvio padrão 1, evitando que atributos de escala maior dominem o modelo.
- Regra de ouro: `fit_transform` no treino, `transform` (sem `fit`) no teste — evita vazamento de dados (*data leakage*).

### Exercício sugerido

Refaça a divisão treino/teste com `test_size=0.3` e compare o novo formato de `X_train`/`X_test` com o obtido anteriormente. As proporções de `status` continuam preservadas com `stratify=y`?
